### 데이터 로드

In [21]:
import pandas as pd

df = pd.read_csv('./강형욱_기초교육.csv')
print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

# 중복 제거
df.drop_duplicates(inplace=True, subset=['video_id'])

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

(전체 데이터 수, 전체 컬럼 수): (153, 7)
(전체 데이터 수, 전체 컬럼 수): (153, 7)


### 기본 전처리

In [2]:
# 누락 및 길이가 짧은 데이터 제거
mask = df['caption'].str.len() > 30
df = df[mask]
df[mask].shape

C:\Users\Playdata\AppData\Local\Temp\ipykernel_25612\86756132.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[mask].shape


(97, 7)

In [5]:
import re

def clean_youtube_captions(text):
    if not isinstance(text, str):
        return ""
    
    # 1. 대괄호 및 소괄호 태그 제거 (예: [음악], (웃음), [박수])
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\(.*?\)', '', text)
    
    # 2. 유튜브 관련 해시태그 및 홍보 문구 제거
    text = re.sub(r'#\S+', '', text)  # #구독, #좋아요 등 제거
    text = text.replace('쿠키영상 있음!', '')
    
    # 3. CSV 파싱 중복 따옴표 정리 (""안쓰럽다"" -> "안쓰럽다")
    text = re.sub(r'""+', '"', text)
    
    # 4. 말줄임표(...) 및 2개 이상 연속된 마침표 제거 (한데... -> 한데, 와.. -> 와)
    text = re.sub(r'\.{2,}', '', text)
    
    # 5. 무의미한 구어체 필러 단어 및 결합 문장부호 완전 제거
    # - (?<=^|\s) : 문자열 시작점 혹은 공백 뒤에 위치하고
    # - [우~?!.,]* : 필러 단어 뒤에 붙은 모음 확장 및 문장부호(?, !, ~, ., , 등)까지 통째로 삼켜 매칭
    # - (?=\s|$) : 공백 혹은 문자열 끝으로 끝나는 standalone 어절인 경우만 매칭 (예: '어떻게'의 '어'는 매칭 안 됨)
    fillers_pattern = r'(^|\s)(?:어|아|음|오|으|에휴|참나|아이고|오구|쉿|저기|그러니까|이제|뭐)[우~?!.,]*(?=\s|$)'
    text = re.sub(fillers_pattern, r'\1', text)
    
    # 6. 남아있는 과도한 다중 문장부호 축소 (예: 앉아!!! -> 앉아!)
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'~+', '~', text)
    
    # 7. 대화 시작 기호(-) 정리 및 불필요한 공백 제거
    text = re.sub(r'\s*-\s*', ' ', text)
    
    # 8. 중복 문구 정제 (연속으로 똑같이 반복되는 어절 제거, 예: "이게 이게" -> "이게")
    words = text.split()
    cleaned_words = []
    prev_word = None
    for word in words:
        if word == prev_word and len(word) > 1:
            continue
        cleaned_words.append(word)
        prev_word = word
    text = " ".join(cleaned_words)
    
    # 9. 이중 공백 제거 및 양끝 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [25]:
df['cleaned_caption'] = df['caption'].apply(clean_youtube_captions)

### LLM 사용 전처리

In [8]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5-mini",
    reasoning_effort="high",        # 논리성 강화
)

In [29]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
너는 영상 캡션 데이터를 정제하는 전문가다.

아래 문장은 음성 인식 또는 자동 생성된 캡션이라
깨진 글자, 반복 표현, 의미 없는 소리, 불필요한 기호가 포함되어 있다.

작업 목표:
- 의미는 유지한다
- 자연스럽고 올바른 한국어 문장으로 복원한다
- 새 정보를 추가하지 않는다
- 설명하지 말고 결과 문장만 출력한다

반드시 지켜야 할 치환 규칙:
- "보듬TV" → "반려견 훈련소"
- "강형욱" → "반려견 훈련사"

주의:
- 위 치환은 문맥과 상관없이 **항상 적용**
- 다른 고유명사는 임의로 바꾸지 않는다
- 치환 사실을 설명하거나 주석을 달지 않는다

입력 문장:
{caption}

정제된 문장:
"""
)


In [30]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

In [31]:
captions = df["cleaned_caption"].tolist()

print(f"captions: {len(captions)}")

captions: 140


In [32]:
inputs = [
    {"caption": caption} for caption in captions if len(caption) > 30
]
len(inputs)

140

In [33]:
from tqdm import tqdm

BATCH_SIZE = 24
MAX_CONCURRENCY = 8 # 최대 8개의 요청을 동시에 날림

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

results = []

for chunk in tqdm(list(chunks(inputs, BATCH_SIZE))):
    results.extend(
        chain.batch(
            chunk,
            config={"max_concurrency": MAX_CONCURRENCY}
        )
    )

100%|██████████| 6/6 [27:36<00:00, 276.01s/it]


In [34]:
cleaned_texts = []

for i in range(len(results)):
    cleaned_texts.append(results[i])

In [ ]:
df["new_cleaned_caption"] = cleaned_texts
df.head()

,playlist_index,video_url,video_id,title,channel,caption,error,cleaned_caption,new_cleaned_caption
0,1,https://www.youtube.com/watch?v=kRsYuNPOzos,kRsYuNPOzos,Whose dog is this? Kang trainer's Beginner dog...,강형욱의 보듬TV,어?! 누구의 누구 개인가요? 얘는? 개는 이제 주인이 사실 명확하지 않아요. 지금...,NaN,누구의 누구 개인가요? 얘는? 개는 이제 주인이 사실 명확하지 않아요. 지금 말도 ...,이 개는 누구 소유인가요? 주인이 명확하지 않습니다. 지금 말도 안 돼요. 얘는 공...
1,2,https://www.youtube.com/watch?v=xl6E2beOjxE,xl6E2beOjxE,Is Jindo dog lame? Kang trainer who requested ...,강형욱의 보듬TV,앉아!!! 보호자님 인정해야 될게... 내 개는 찌질해요. 거주하는 환경에 따라 반...,NaN,앉아! 보호자님 인정해야 될게 내 개는 찌질해요. 거주하는 환경에 따라 반려견과 살...,거주하는 환경에 따라 반려견과 살아가는 모습도 달라집니다. 반려견 훈련사가 같은 지...
2,3,https://www.youtube.com/watch?v=vftVEbbgHZE,vftVEbbgHZE,"""Are you crazy?"" The reason for Kang Hyung-woo...",강형욱의 보듬TV,몇 개월 됐어요? 이제 3개월하고 보름 조금 지났어요. 1년 뒤에 아마 개훌륭 나오...,NaN,몇 개월 됐어요? 이제 3개월하고 보름 조금 지났어요. 1년 뒤에 아마 개훌륭 나오...,몇 개월 됐어요? \n이제 3개월하고 보름 조금 지났어요. \n1년 뒤면 아마 ...
3,4,https://www.youtube.com/watch?v=uXLh6C6iccM,uXLh6C6iccM,"A genius dog coveted by Kang Hyung-Wook, a 3-m...",강형욱의 보듬TV,자네 이름이 뭔가 영재가 아니고 천잰데 얘는 와.. 얘는 어 국가를 위해서 일해야죠...,NaN,자네 이름이 뭔가 영재가 아니고 천잰데 얘는 와 얘는 국가를 위해서 일해야죠 먹는 ...,"자네 이름이 뭐죠? 영재라기보다 천재 같은데, 이 친구는 국가를 위해 일할 수 있을..."
4,5,https://www.youtube.com/watch?v=JC_PMiAtPMo,JC_PMiAtPMo,How to play nosework from the very basic step ...,강형욱의 보듬TV,노즈워크 놀이방법 A to Z까지 뭐 여러 가지 좋은 방법들이 있어요 냄새 맡는 거...,NaN,노즈워크 놀이방법 A to Z까지 뭐 여러 가지 좋은 방법들이 있어요 냄새 맡는 거...,노즈워크 놀이 방법에는 여러 가지가 있습니다. 냄새 맡는 활동은 아주 좋은데 조절력...


### 저장

In [39]:
df[['video_id', 'video_url', 'channel', 'title', 'new_cleaned_caption']].to_csv('./basic_instruction.csv', index=False, header=True)

---

In [1]:
import pandas as pd

df = pd.read_csv('./설채현의수의학.csv')
print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

# 중복 제거
df.drop_duplicates(inplace=True, subset=['video_id'])

print(f"(전체 데이터 수, 전체 컬럼 수): {df.shape}")

(전체 데이터 수, 전체 컬럼 수): (101, 7)
(전체 데이터 수, 전체 컬럼 수): (101, 7)


In [3]:
# 누락 및 길이가 짧은 데이터 제거
mask = df['caption'].str.len() > 30
df = df[mask]
df[mask].shape

(97, 7)

In [6]:
df['cleaned_caption'] = df['caption'].apply(clean_youtube_captions)

In [7]:
df

,playlist_index,video_url,video_id,title,channel,caption,error,cleaned_caption
0,1,https://www.youtube.com/watch?v=ZHpM_Scrpxk,ZHpM_Scrpxk,강아지 유산균으로 입냄새도 없앨 수 있다고요? 🤭 l 강아지 입냄새 없애는 법 l ...,설채현의 놀로와,[음악] 안녕하세요 설치는 여러분 설치연수의 삽니다 이번 시간에는 우리 반려견 보호...,NaN,안녕하세요 설치는 여러분 설치연수의 삽니다 이번 시간에는 우리 반려견 보호자분들이 ...
2,3,https://www.youtube.com/watch?v=Mq_WN5BRWWk,Mq_WN5BRWWk,Winter Walk: Should I Dress My Dog Up or Not?!...,설채현의 놀로와,안녕하세요 독설 tv 소천 수 있습니다 네오는 공간이 밖에서도 오셨어요 그동안 약간...,NaN,안녕하세요 독설 tv 소천 수 있습니다 네오는 공간이 밖에서도 오셨어요 그동안 약간...
3,4,https://www.youtube.com/watch?v=UyjcuBZIv68,UyjcuBZIv68,[Everything About Dog Neutering Part 1] The Se...,설채현의 놀로와,반려견 에 대한 궁금증을 솔직하고 독하게 알려드리는 설치 할 수 있습니다 아 오늘 ...,NaN,반려견 에 대한 궁금증을 솔직하고 독하게 알려드리는 설치 할 수 있습니다 오늘 주제...
4,5,https://www.youtube.com/watch?v=93xZNcmwdOg,93xZNcmwdOg,Patellar Luxation Prevention: From Walking Tip...,설채현의 놀로와,안녕하세요 강아지에 대한 궁금증을 솔직하고 복하게 알려드리는 독설 2b 설수현 수입...,NaN,안녕하세요 강아지에 대한 궁금증을 솔직하고 복하게 알려드리는 독설 2b 설수현 수입...
5,6,https://www.youtube.com/watch?v=92EByzD_Dt4,92EByzD_Dt4,What to do when your dog has a foreign object ...,설채현의 놀로와,그 아내에 대한 정보를 도 카도 솔직하게 알려 드리는 독설 tv 설치 운 수 있습니...,NaN,그 아내에 대한 정보를 도 카도 솔직하게 알려 드리는 독설 tv 설치 운 수 있습니...
...,...,...,...,...,...,...,...,...
96,97,https://www.youtube.com/watch?v=cQ21fb3hk0I,cQ21fb3hk0I,"If you create a habit, success is guaranteed 💯...",설채현의 놀로와,"아, 사실 전문가라고 하는 사람들조차 잘 모르고 있는 것이 있습니다. 화장실의 가장...",NaN,사실 전문가라고 하는 사람들조차 잘 모르고 있는 것이 있습니다. 화장실의 가장 기본...
97,98,https://www.youtube.com/watch?v=VvgET_oS9hY,VvgET_oS9hY,Can Dog Walking Be Banned in Apartment Buildin...,설채현의 놀로와,트집 잡아서 강아지 꼴보기 싫는 사람들이 한번 투표를 시작한 거. 난 욕 먹어도 돼...,NaN,트집 잡아서 강아지 꼴보기 싫는 사람들이 한번 투표를 시작한 거. 난 욕 먹어도 돼...
98,99,https://www.youtube.com/watch?v=RZWxxahvTc4,RZWxxahvTc4,동물 학대 처벌의 현실🤬 비비탄 사건부터 입마개 논란까지! l 이슈체크 l 설채현 ...,설채현의 놀로와,네 마리를 향해서 한 마에 비비한 총을 난사했다. 그래서 한 마리가 사망. 근데 이...,NaN,네 마리를 향해서 한 마에 비비한 총을 난사했다. 그래서 한 마리가 사망. 근데 이...
99,100,https://www.youtube.com/watch?v=2OaVR46dsWM,2OaVR46dsWM,🚨Overexcited🚨 Dogs Could Be Having a Brain Pro...,설채현의 놀로와,제가 계속 답답한 경우가 뭐냐면 어차피 지금 아무리 가르쳐봤자 듣지 않는 아이들한테...,NaN,제가 계속 답답한 경우가 뭐냐면 어차피 지금 아무리 가르쳐봤자 듣지 않는 아이들한테...


In [10]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template(
"""
너는 영상 캡션 데이터를 정제하는 전문가다.

아래 문장은 음성 인식 또는 자동 생성된 캡션이라
깨진 글자, 반복 표현, 의미 없는 소리, 불필요한 기호가 포함되어 있다.

작업 목표:
- 의미는 유지한다
- 자연스럽고 올바른 한국어 문장으로 복원한다
- 새 정보를 추가하지 않는다
- 설명하지 말고 결과 문장만 출력한다

반드시 지켜야 할 치환 규칙:
- "놀로와" → "반려동물병원"
- "설채현", "설치연", "소천수", "설수현" 등 → "수의사"

주의:
- 위 치환은 문맥과 상관없이 **항상 적용**
- 다른 고유명사는 임의로 바꾸지 않는다
- 치환 사실을 설명하거나 주석을 달지 않는다

입력 문장:
{caption}

정제된 문장:
"""
)


In [13]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

In [14]:
captions = df["cleaned_caption"].tolist()

print(f"captions: {len(captions)}")

inputs = [
    {"caption": caption} for caption in captions if len(caption) > 30
]
len("inputs:", inputs)

captions: 97


97

In [16]:
from tqdm import tqdm

BATCH_SIZE = 24
MAX_CONCURRENCY = 8 # 최대 8개의 요청을 동시에 날림

def chunks(lst, size):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

results = []

for chunk in tqdm(list(chunks(inputs, BATCH_SIZE))):
    results.extend(
        chain.batch(
            chunk,
            config={"max_concurrency": MAX_CONCURRENCY}
        )
    )

cleaned_texts = []

for i in range(len(results)):
    cleaned_texts.append(results[i])

df["new_cleaned_caption"] = cleaned_texts
df[['video_id', 'video_url', 'channel', 'title', 'new_cleaned_caption']].to_csv('./vet_knowledge.csv', index=False, header=True)

100%|██████████| 5/5 [31:26<00:00, 377.28s/it]
